In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import dukascopy_python.compat as duka
from dukascopy_python.compat.instruments import (
    INSTRUMENT_FX_MAJORS_EUR_USD,
)

In [2]:
df_ask = duka.fetch(
    INSTRUMENT_FX_MAJORS_EUR_USD,
    duka.INTERVAL_HOUR_1,
    duka.OFFER_SIDE_ASK,          # đổi BID -> ASK
    datetime(2025, 1, 1, tzinfo=timezone.utc),
    datetime(2026, 1, 1, tzinfo=timezone.utc),
)
df_ask.to_parquet("../data/raw/EURUSD_H1_ask.parquet")   # lưu thẳng parquet, khỏi lo parse lại

In [3]:
df = pd.read_parquet("../data/raw/EURUSD_H1.parquet")

In [4]:
df["ret"] = df["close"].pct_change()
df["range"] = df["high"] - df["low"]
df

,open,high,low,close,volume,ret,range
timestamp,,,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22,NaN,0.00094
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37,-0.000415,0.00101
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37,0.000126,0.00092
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45,0.000560,0.00164
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47,0.001265,0.00135
...,...,...,...,...,...,...,...
2025-12-31 17:00:00+00:00,1.17346,1.17390,1.17312,1.17358,3662.76,0.000094,0.00078
2025-12-31 18:00:00+00:00,1.17357,1.17386,1.17302,1.17357,3944.61,-0.000009,0.00084
2025-12-31 19:00:00+00:00,1.17356,1.17432,1.17356,1.17407,3321.68,0.000426,0.00076


In [5]:
avg_hour_range = df["range"].groupby(df.index.hour).mean() / 0.0001
avg_hour_ret = abs(df["ret"]).groupby(df.index.hour).mean() / 0.0001
result = pd.concat([avg_hour_range,avg_hour_ret], axis="columns")  
result

,range,ret
timestamp,,
0,14.333077,6.748503
1,13.932692,5.834818
2,10.968462,4.352324
3,10.002692,4.338132
4,9.212308,4.051846
5,11.245769,4.834920
6,15.996538,6.984422
7,20.659231,9.557725
8,19.202703,8.021376


In [6]:
z = avg_hour_range.max() / avg_hour_range.min()
print(z)

2.8956581352833886


In [7]:
print(result.sort_values(by="ret",ascending=False))

               range        ret
timestamp                      
14         24.462162  10.977947
15         23.296139   9.890514
13         23.695753   9.835287
12         22.257529   9.688574
7          20.659231   9.557725
8          19.202703   8.021376
16         17.355212   7.192835
11         16.683012   7.087935
6          15.996538   6.984422
0          14.333077   6.748503
10         15.467568   6.610443
9          15.847287   6.524206
18         14.682625   6.258243
17         15.166409   5.965693
1          13.932692   5.834818
19         12.139382   5.010343
22          9.986154   4.931869
5          11.245769   4.834920
23         10.270385   4.570730
2          10.968462   4.352324
3          10.002692   4.338132
21          8.447876   4.145972
20         10.679537   4.075133
4           9.212308   4.051846


In [8]:
hour_ny = df.index.tz_convert("America/New_York").hour

df["is_asia"]     = (hour_ny >= 19) | (hour_ny < 4)   # qua nửa đêm -> |
df["is_london"]   = (hour_ny >= 3)  & (hour_ny < 12)  # khoảng liền -> &
df["is_new_york"] = (hour_ny >= 8)  & (hour_ny < 17)

sessions = ["is_asia", "is_london", "is_new_york"]
summary = pd.DataFrame({
    s: {
        "n": df[s].sum(),
        "range_pip": df.loc[df[s], "range"].mean() * 10_000,
        "abs_ret_bp": df.loc[df[s], "ret"].abs().mean() * 10_000,
    }
    for s in sessions
}).T
summary

,n,range_pip,abs_ret_bp
is_asia,2339.0,13.273065,5.846831
is_london,2330.0,20.270644,8.684874
is_new_york,2331.0,17.968211,7.565658


In [9]:
df

,open,high,low,close,volume,ret,range,is_asia,is_london,is_new_york
timestamp,,,,,,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22,NaN,0.00094,False,False,False
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37,-0.000415,0.00101,False,False,False
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37,0.000126,0.00092,True,False,False
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45,0.000560,0.00164,True,False,False
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47,0.001265,0.00135,True,False,False
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 17:00:00+00:00,1.17346,1.17390,1.17312,1.17358,3662.76,0.000094,0.00078,False,False,True
2025-12-31 18:00:00+00:00,1.17357,1.17386,1.17302,1.17357,3944.61,-0.000009,0.00084,False,False,True
2025-12-31 19:00:00+00:00,1.17356,1.17432,1.17356,1.17407,3321.68,0.000426,0.00076,False,False,True


In [10]:
week_summary = df.groupby(df.index.tz_convert("America/New_York").day_of_week).agg(
    range_pip = ("range", "mean"),
    abs_ret = ("ret", lambda x: x.abs().mean()),
    n = ("range","size")
)

week_summary[["range_pip","abs_ret"]] *= 10000
week_summary

,range_pip,abs_ret,n
timestamp,,,
0,14.800962,6.476154,1247
1,14.408974,6.132456,1248
2,15.428045,6.664140,1248
3,15.668476,6.599114,1234
4,17.156674,7.130892,884
6,12.932692,6.451477,364


In [11]:
ny_time = df.index.tz_convert("America/New_York")
trading_dow = (ny_time + pd.Timedelta(hours=7)).day_of_week   

week_summary = df.groupby(trading_dow).agg(
    range_pip=("range", "mean"),
    abs_ret_bp=("ret", lambda x: x.abs().mean()),
    n=("range", "size"),
)
week_summary[["range_pip", "abs_ret_bp"]] *= 10_000
week_summary

,range_pip,abs_ret_bp,n
timestamp,,,
0,15.457097,6.999664,1247
1,14.561699,6.216587,1248
2,15.249599,6.558673,1248
3,15.649919,6.666795,1234
4,15.313141,6.364055,1248


In [12]:
df_merge = pd.merge(df,df_ask,left_index=True,right_index=True, how="inner",suffixes=("_bid", "_ask"))
df_merge["spread"] = df_merge["close_ask"] - df_merge["close_bid"]
df_merge["spread"] *= 10000
df_merge

,open_bid,high_bid,low_bid,close_bid,volume_bid,ret,range,is_asia,is_london,is_new_york,open_ask,high_ask,low_ask,close_ask,volume_ask,spread
timestamp,,,,,,,,,,,,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22,NaN,0.00094,False,False,False,1.03591,1.03612,1.03515,1.03583,1837.760,3.3
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37,-0.000415,0.00101,False,False,False,1.03582,1.03590,1.03468,1.03511,2162.879,0.4
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37,0.000126,0.00092,True,False,False,1.03511,1.03567,1.03476,1.03523,5531.480,0.3
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45,0.000560,0.00164,True,False,False,1.03523,1.03611,1.03444,1.03581,8160.490,0.3
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47,0.001265,0.00135,True,False,False,1.03581,1.03714,1.03580,1.03712,4256.690,0.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 17:00:00+00:00,1.17346,1.17390,1.17312,1.17358,3662.76,0.000094,0.00078,False,False,True,1.17353,1.17392,1.17316,1.17363,3690.570,0.5
2025-12-31 18:00:00+00:00,1.17357,1.17386,1.17302,1.17357,3944.61,-0.000009,0.00084,False,False,True,1.17361,1.17392,1.17305,1.17361,4421.420,0.4
2025-12-31 19:00:00+00:00,1.17356,1.17432,1.17356,1.17407,3321.68,0.000426,0.00076,False,False,True,1.17363,1.17434,1.17363,1.17409,2652.440,0.2


In [13]:
df_merge["spread"].mean()

np.float64(0.6632931726907627)

In [14]:
result = df_merge["spread"].groupby(df_merge.index.tz_convert("America/New_York").hour).mean()
print(result.sort_values(ascending=False))

timestamp
17    2.967692
16    2.761390
18    0.704231
9     0.643629
19    0.533462
1     0.513077
4     0.496899
3     0.481853
21    0.476154
2     0.473462
13    0.461776
15    0.457915
20    0.456154
0     0.448077
23    0.441923
22    0.427692
5     0.422008
8     0.411583
11    0.410811
12    0.407722
7     0.399228
14    0.388417
6     0.386873
10    0.343243
Name: spread, dtype: float64


In [15]:
(df_merge["spread"] < 0).any()

np.False_

In [16]:
daytime_summary = df_merge.groupby([df_merge.index.tz_convert("America/New_York").hour, df_merge.index.tz_convert("America/New_York").day_of_week])["spread"].mean()

In [17]:
daytime_summary.unstack()

timestamp,0,1,2,3,4,6
timestamp,,,,,,
0,0.400000,0.469231,0.434615,0.484615,0.451923,NaN
1,0.496154,0.478846,0.513462,0.521154,0.555769,NaN
2,0.398077,0.405769,0.390385,0.634615,0.538462,NaN
3,0.471154,0.457692,0.465385,0.466667,0.548077,NaN
4,0.433333,0.565385,0.513462,0.458824,0.511538,NaN
5,0.442308,0.426923,0.403846,0.384314,0.451923,NaN
6,0.382692,0.388462,0.396154,0.366667,0.400000,NaN
7,0.415385,0.382692,0.400000,0.354902,0.442308,NaN
8,0.403846,0.394231,0.421154,0.405882,0.432692,NaN


In [18]:
df_merge

,open_bid,high_bid,low_bid,close_bid,volume_bid,ret,range,is_asia,is_london,is_new_york,open_ask,high_ask,low_ask,close_ask,volume_ask,spread
timestamp,,,,,,,,,,,,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22,NaN,0.00094,False,False,False,1.03591,1.03612,1.03515,1.03583,1837.760,3.3
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37,-0.000415,0.00101,False,False,False,1.03582,1.03590,1.03468,1.03511,2162.879,0.4
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37,0.000126,0.00092,True,False,False,1.03511,1.03567,1.03476,1.03523,5531.480,0.3
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45,0.000560,0.00164,True,False,False,1.03523,1.03611,1.03444,1.03581,8160.490,0.3
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47,0.001265,0.00135,True,False,False,1.03581,1.03714,1.03580,1.03712,4256.690,0.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 17:00:00+00:00,1.17346,1.17390,1.17312,1.17358,3662.76,0.000094,0.00078,False,False,True,1.17353,1.17392,1.17316,1.17363,3690.570,0.5
2025-12-31 18:00:00+00:00,1.17357,1.17386,1.17302,1.17357,3944.61,-0.000009,0.00084,False,False,True,1.17361,1.17392,1.17305,1.17361,4421.420,0.4
2025-12-31 19:00:00+00:00,1.17356,1.17432,1.17356,1.17407,3321.68,0.000426,0.00076,False,False,True,1.17363,1.17434,1.17363,1.17409,2652.440,0.2
